# Modeling
steps:
- Preprocessing the test set like the train data
- Scaling the data (test and train)
- function for model_evaluation
- function for model building
- 

In [29]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn import preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict, cross_validate
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
from sklearn.metrics import fbeta_score, make_scorer, f1_score, accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import RandomizedSearchCV

import importlib
import preprocessing_mel
importlib.reload(preprocessing_mel)
from preprocessing_mel import preprocess_minimal

RSEED = 42

## Scaling the data

## Modeling

In [30]:
df = pd.read_csv('data/Train.csv')

In [31]:
X = df.drop(columns=['target', 'target_min', 'target_max', 
                     'target_variance', 'target_count'])
y = df['target']

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RSEED,)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [34]:
# Train: Fit transformers on training data
X_train_proc, y_train_proc = preprocess_minimal(
    df=X_train,
    y=y_train,
    fit_data=X_train  #  Learn from training data
)

# Test: Apply training transformers
X_test_proc, y_test_proc = preprocess_minimal(
    df=X_test,
    y=y_test,
    fit_data=X_train  #  Use training statistics (NO LEAKAGE!)
)

# Drop non-numeric columns
cols_to_drop = ['Date', 'Place_ID', 'Place_ID X Date']
X_train_final = X_train_proc.drop(columns=cols_to_drop, errors='ignore')
X_test_final = X_test_proc.drop(columns=cols_to_drop, errors='ignore')

y_train_final = y_train_proc
y_test_final = y_test_proc

print(f"✅ X_train: {X_train_final.shape}, y_train: {y_train_final.shape}")
print(f"✅ X_test: {X_test_final.shape}, y_test: {y_test_final.shape}")

✅ X_train: (24445, 74), y_train: (24445,)
✅ X_test: (6112, 74), y_test: (6112,)


In [35]:
X_train_final.isna().sum()

precipitable_water_entire_atmosphere    0
relative_humidity_2m_above_ground       0
specific_humidity_2m_above_ground       0
temperature_2m_above_ground             0
u_component_of_wind_10m_above_ground    0
                                       ..
L3_CH4_aerosol_optical_depth            0
L3_CH4_sensor_azimuth_angle             0
L3_CH4_sensor_zenith_angle              0
L3_CH4_solar_azimuth_angle              0
L3_CH4_solar_zenith_angle               0
Length: 74, dtype: int64

In [36]:

list_of_clf = [LinearRegression(),
               KNeighborsRegressor(),
               RandomForestRegressor(),
               GradientBoostingRegressor()
               ]

tscv = TimeSeriesSplit(n_splits=5)

scorers = {
    'r2': 'r2',
    'rmse': make_scorer(mean_squared_error, greater_is_better=False, squared=False)
}

def model_evaluation(estimator, scoring, X_train_final, y_train_final, cv=tscv):
    return cross_validate(estimator, X_train_final, y_train_final, cv=cv, scoring=scoring, return_train_score=False)

for reg in list_of_clf:
    results = model_evaluation(reg, scorers, X_train_final, y_train_final, cv=5)
    print(reg)
   
    r2_mean, r2_std = results['test_r2'].mean(), results['test_r2'].std()
    rmse_mean, rmse_std = (-results['test_rmse']).mean(), (-results['test_rmse']).std()  # Vorzeichen umdrehen
    print(f'R²   (mean ± std): {r2_mean:.3f} ± {r2_std:.3f}')
    print(f'RMSE (mean ± std): {rmse_mean:.3f} ± {rmse_std:.3f}')
    print('----'*10)

LinearRegression()
R²   (mean ± std): 0.374 ± 0.018
RMSE (mean ± std): 37.052 ± 0.830
----------------------------------------
KNeighborsRegressor()
R²   (mean ± std): 0.415 ± 0.025
RMSE (mean ± std): 35.810 ± 0.923
----------------------------------------
RandomForestRegressor()
R²   (mean ± std): 0.583 ± 0.028
RMSE (mean ± std): 30.245 ± 1.272
----------------------------------------
GradientBoostingRegressor()
R²   (mean ± std): 0.490 ± 0.024
RMSE (mean ± std): 33.451 ± 1.159
----------------------------------------


## Hyperparameter Tuning

- LinearRegression - GradientDescent
- KNN - GridSearchCV?
- Decision Tree - GridSearchCV?

## KNNReg parametertuning

In [37]:
KNeighborsRegressor().get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [41]:
# Defining parameter grid (as dictionary)
param_grid = {"n_neighbors" : [3,5,10], #this actually defines the model you use
              "weights" : ["uniform", "distance"],
              "p" : [1, 2],
              "algorithm": ["ball_tree"],
              "leaf_size": [30],
              "weights": ["uniform", "distance"],
             }

# Instantiate gridsearch and define the metric to optimize 
gs = GridSearchCV(KNeighborsRegressor(), param_grid, scoring='neg_root_mean_squared_error',
                  cv=5, verbose=2, n_jobs=-1)

# Fit gridsearch object to data.. also lets see how long it takes

gs.fit(X_train_final, y_train_final)



Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=uniform; total time=  15.9s
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=distance; total time=  16.0s
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=uniform; total time=  16.3s
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=uniform; total time=  21.3s
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=uniform; total time=  22.1s
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=uniform; total time=  22.1s
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=distance; total time=  22.5s
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=distance; total time=  22.7s
[CV] END algorithm=ball_tree, leaf_size=30, n_neighbors=3, p=1, weights=distance; total time=  13.4s
[CV] END algorithm=ball_tree, leaf_

GridSearchCV(cv=5, estimator=KNeighborsRegressor(), n_jobs=-1,
             param_grid={'algorithm': ['ball_tree'], 'leaf_size': [30],
                         'n_neighbors': [3, 5, 10], 'p': [1, 2],
                         'weights': ['uniform', 'distance']},
             scoring='neg_root_mean_squared_error', verbose=2)

In [42]:
# Best score
print('Best score:', -round(gs.best_score_, 3))

# Best parameters
print('Best parameters:', gs.best_params_)

Best score: 34.804
Best parameters: {'algorithm': 'ball_tree', 'leaf_size': 30, 'n_neighbors': 10, 'p': 1, 'weights': 'distance'}


In [49]:
# Assigning the fitted KNNClassifier model with best parameter combination to a new variable knn_best
knn_best_gs = gs.best_estimator_

# Making predictions on the train set
y_pred_knn = knn_best_gs.predict(X_test_final)

print("R2 on train:", round(r2_score(y_test_final, y_pred_knn),3))

R2 on train: 0.465


In [45]:
def print_pretty_summary(name, y_actual, y_pred):
    print(name)
    print('=======================')
    
    r2 = r2_score(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
    
    print(f"R²:   {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print()

In [46]:
print_pretty_summary("KNNreg",y_test, y_pred_knn)

KNNreg
R²:   0.465
RMSE: 34.308



## RandomForestReg parametertuning

In [47]:
RandomForestRegressor().get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': 1.0,
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': None,
 'verbose': 0,
 'warm_start': False}

In [52]:
# --- CV und Scorer definieren ---
tscv = TimeSeriesSplit(n_splits=5)
rmse_scorer = make_scorer(mean_squared_error, greater_is_better=False, squared=False)

# --- Modell ---
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# --- Parameter-Raum ---
param_dist = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"],
    "bootstrap": [True],
}

# --- Randomized Search ---
rand_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,                  # Anzahl zufälliger Kombinationen
    scoring=rmse_scorer,
    cv=tscv,
    n_jobs=-1,
    random_state=42,
    verbose=2,
    refit=True
)

rand_search.fit(X_train_final, y_train_final)

# --- Ergebnisse ---
print("\nBeste Parameterkombination:")
print(rand_search.best_params_)
print("Bester RMSE (CV):", -rand_search.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


/Users/melaniesass/neueFische/ds-ml-project-urban-air-pollution/.venv/lib/python3.11/site-packages/sklearn/model_selection/_search.py:305: UserWarning: The total space of parameters 24 is smaller than n_iter=30. Running 24 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   4.9s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=400; total time=   9.8s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  10.6s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=200; total time=   4.7s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  16.1s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=400; total time=  21.3s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  22.8s
[CV] END bootstrap=True, max_depth

/Users/melaniesass/neueFische/ds-ml-project-urban-air-pollution/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=200; total time=   8.1s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=400; total time=  29.2s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=400; total time=   7.5s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=200; total time=  14.1s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=400; total time=  58.8s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=200; total time=  19.7s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=200; total time=   3.8s
[CV] END bootstrap=True, max_depth

In [54]:
print("Best RMSE (CV):", -round(rand_search.best_score_, 3))
print("Best Parameters:", rand_search.best_params_)

Best RMSE (CV): 32.433
Best Parameters: {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None, 'bootstrap': True}


In [55]:
# Assigning the fitted random forest classifier model with best parameter combination to a new variable rf_best_gs
rf_best_rs = rand_search.best_estimator_

# Making predictions on the train set
y_pred_rf = rf_best_rs.predict(X_test_final)

print("R2 on train:", round(r2_score(y_test_final, y_pred_rf),3))


R2 on train: 0.565


In [57]:
print_pretty_summary("Random Forest",y_test_final, y_pred_rf)

Random Forest
R²:   0.565
RMSE: 30.951



## Evaluation

of different models

## Error Analysis